In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def generate_comparison_plots(data, dataset_name):
    
    # Define unique elements for plotting
    interventions = [('p', '$p^{new}$'), 
                     ('rho', '$\\mu_{\\rho}$ of $P(\\rho)$'), 
                     ('gamma', '$\\mu_{\\gamma}$ of $P(\\gamma)$')]
    
    models = sorted(data['model'].unique())
    
    # Define colors and markers for each model
    colors = {'MHSA': 'tab:blue', 'Mamba': 'tab:green', 'LSTM': 'tab:orange'}
    markers = {'MHSA': 'o', 'Mamba': 'o', 'LSTM': 'o'}

    # Create a 3x4 subplot grid
    fig, axes = plt.subplots(3, 4, figsize=(24, 15))
    fig.suptitle(f'Model Robustness on Interventional {dataset_name} Dataset', fontsize=20, fontweight='bold')

    # Define the correct order for metrics
    metrics_to_plot = [('Acc@1', 'Acc@1 (%)'), 
                       ('Acc@5', 'Acc@5 (%)'),
                       ('F1', 'F1 Score (%)'), 
                       ('MRR', 'MRR (%)')]
    
    # --- CHANGE 1: Define the column titles you want ---
    column_titles = ['Acc@1', 'Acc@5', 'F1-Score', 'MRR']

    subplot_labels = [['(a1)', '(a2)', '(a3)', '(a4)'], 
                      ['(b1)', '(b2)', '(b3)', '(b4)'], 
                      ['(c1)', '(c2)', '(c3)', '(c4)']]

    for i, (intervention_code, intervention_label) in enumerate(interventions):
        # --- FIX 1: Use .str.lower() for case-insensitive matching ---
        intervention_df = data[data['Intervention'].str.lower() == intervention_code]
        
        for j, (metric_code, metric_label) in enumerate(metrics_to_plot):
            ax = axes[i, j]

            # --- CHANGE 2: Add titles to the top row of subplots (when i is 0) ---
            if i == 0:
                ax.set_title(column_titles[j], fontsize=16, fontweight='bold')

            for model in models:
                model_df = intervention_df[intervention_df['model'] == model].sort_values('value')
                
                # Check if there is data to plot
                if not model_df.empty:
                    # --- FIX 2: Use direct dictionary access for robust color mapping ---
                    ax.plot(model_df['value'], model_df[metric_code], 
                            marker=markers[model], 
                            color=colors[model], 
                            label=model,
                            # alpha=0.75,
                            linestyle='-')

            ax.set_xlabel(intervention_label, fontsize=12)
            ax.set_ylabel(metric_label, fontsize=12)
            
            # Remove duplicate labels in legend
            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            ax.legend(by_label.values(), by_label.keys())
            
            ax.grid(True, linestyle='--', alpha=0.3)
            ax.text(0.05, 0.05, subplot_labels[i][j], transform=ax.transAxes, 
                    fontsize=18, fontweight='bold', va='bottom')

    # Adjust layout to make space for titles
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjusted top margin for suptitle
    filename = f'comparison_plot_{dataset_name}.png'
    plt.savefig(filename, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Corrected plot saved as {filename}")

# Example usage (assuming you have a DataFrame named 'your_dataframe')
# generate_comparison_plots(your_dataframe, 'YourDatasetName')


In [ ]:
# 文件名: generate_tables_corrected.py

import pandas as pd
import io

def apply_highlighting(df):
    """
    对DataFrame进行处理，添加用于高亮的字符串列。
    - 比较每个干预点上三个模型的指标。
    - 最佳值加粗，次佳值加下划线。
    - 所有数值格式化为三位小数。
    """
    metrics = ['Acc@1', 'Acc@3', 'Acc@5', 'Acc@10', 'F1', 'MRR']

    # 1. 首先，为每个指标创建一个格式化后的字符串列（保留三位小数）
    for metric in metrics:
        df[f'{metric}_str'] = df[metric].apply(lambda x: f'{x:.3f}')

    # 2. 按干预类型和值进行分组，以便比较模型
    grouped = df.groupby(['Intervention', 'value'])

    # 3. 遍历每个分组，找出最佳和次佳的指标
    for _, group_df in grouped:
        # 仅在有3个模型可供比较时才进行高亮
        if len(group_df) == 3:
            for metric in metrics:
                # 对当前指标进行降序排序以找到最佳和次佳值
                sorted_values = group_df[metric].sort_values(ascending=False)
                
                best_val = sorted_values.iloc[0]
                second_val = sorted_values.iloc[1]
                
                # 找到最佳和次佳值所在的行索引
                best_indices = group_df[group_df[metric] == best_val].index
                second_indices = group_df[group_df[metric] == second_val].index
                
                # 在原始DataFrame中更新对应的字符串列，添加LaTeX格式
                df.loc[best_indices, f'{metric}_str'] = f'\\textbf{{{best_val:.3f}}}'
                # 确保最佳值和次佳值不同时才添加下划线
                if best_val != second_val:
                    df.loc[second_indices, f'{metric}_str'] = f'\\underline{{{second_val:.3f}}}'
                
    return df

def generate_latex_table(df):
    """
    根据处理好的DataFrame生成最终的LaTeX表格代码。
    """
    dataset_name = df['dataset'].iloc[0]
    models = ['LSTM', 'MHSA', 'Mamba']
    
    # 定义您指定的Intervention顺序和对应的显示名称
    intervention_map = {
        'p': '$p^{\\text{new}}$',
        'rho': '$\\mu$ of $P(\\rho)$',
        'gamma': '$\\mu$ of $P(\\gamma)$'
    }
    intervention_order = ['p', 'rho', 'gamma']

    # --- 开始构建LaTeX字符串 ---
    latex_string = f"\\begin{{table*}}[htbp]\n"
    latex_string += f"\\centering\n"
    latex_string += f"\\caption{{Performance on the Intervened {dataset_name} dataset.}}\n"
    latex_string += f"\\label{{tab:{dataset_name.lower().replace('-', '')}}}\n"
    latex_string += "\\small\n"
    latex_string += "\\begin{tabular}{@{}lllcccccc@{}}\n"
    latex_string += "\\toprule\n"
    latex_string += "Model & Intervention & value & Acc@1 & Acc@3 & Acc@5 & Acc@10 & F1 & MRR \\\\ \n"
    latex_string += "\\midrule\n"

    for i, model in enumerate(models):
        if i > 0:
            latex_string += "\\midrule\n"
        
        model_df = df[df['model'] == model]
        num_model_rows = len(model_df)
        
        latex_string += f"\\multirow{{{num_model_rows}}}{{*}}{{{model}}}"

        first_intervention = True
        for intervention_key in intervention_order:
            intervention_df = model_df[model_df['Intervention'] == intervention_key].sort_values(by='value')
            num_intervention_rows = len(intervention_df)

            if num_intervention_rows > 0:
                if not first_intervention:
                    latex_string += "\\cline{2-9}\n"
                else:
                    first_intervention = False
                
                # *** CORRECTED LOGIC FOR ROW GENERATION STARTS HERE ***
                for i_row, (_, row) in enumerate(intervention_df.iterrows()):
                    if i_row == 0:
                        intervention_label = intervention_map[intervention_key]
                        latex_string += f" & \\multirow{{{num_intervention_rows}}}{{*}}{{{intervention_label}}}"
                    else:
                        # Add empty placeholders for Model and Intervention columns
                        latex_string += " & "
                    
                    # Add value and metric columns
                    latex_string += f" & {row['value']:.2f}"
                    for metric in ['Acc@1', 'Acc@3', 'Acc@5', 'Acc@10', 'F1', 'MRR']:
                        latex_string += f" & {row[f'{metric}_str']}"
                    latex_string += " \\\\ \n"
                # *** CORRECTED LOGIC ENDS HERE ***
    
    latex_string += "\\bottomrule\n"
    latex_string += "\\end{tabular}\n"
    latex_string += "\\end{table*}\n"
    
    return latex_string

# --- 主程序 ---
if __name__ == "__main__":
    # 1. 定义您的最终数据
    dtepr_data_string = """
model,dataset,Intervention,value,Acc@1,Acc@3,Acc@5,Acc@10,F1,MRR
LSTM,DT-EPR,gamma,0.1,59.071,61.978,62.477,63.108,45.675,60.835
LSTM,DT-EPR,gamma,0.25,80.889,83.117,83.427,83.808,73.722,82.175
LSTM,DT-EPR,gamma,0.5,92.258,93.653,93.92,94.214,89.581,93.074
LSTM,DT-EPR,gamma,0.75,94.027,95.29,95.625,96.019,92.319,94.797
LSTM,DT-EPR,gamma,0.9,92.871,94.301,94.741,95.518,91.01,93.837
LSTM,DT-EPR,p,0.1,86.941,89.156,89.485,89.827,82.214,88.192
LSTM,DT-EPR,p,0.25,70.449,73.785,74.31,74.947,60.063,72.392
LSTM,DT-EPR,p,0.5,45.149,48.325,48.906,49.794,29.61,47.161
LSTM,DT-EPR,p,0.75,20.201,22.978,23.574,24.448,7.802,22.094
LSTM,DT-EPR,p,0.9,6.215,7.739,8.247,9.093,1.131,7.522
LSTM,DT-EPR,rho,0.1,71.088,76.393,78.139,80.69,63.472,74.529
LSTM,DT-EPR,rho,0.25,83.082,85.497,86.001,86.78,77.363,84.564
LSTM,DT-EPR,rho,0.5,77.094,79.522,79.907,80.365,68.634,78.522
LSTM,DT-EPR,rho,0.75,69.778,72.419,72.83,73.363,58.876,71.347
LSTM,DT-EPR,rho,0.9,64.699,67.301,67.734,68.288,52.383,66.27
MHSA,DT-EPR,gamma,0.1,59.01,61.966,62.554,63.345,45.981,60.874
MHSA,DT-EPR,gamma,0.25,80.923,83.131,83.449,83.854,73.814,82.21
MHSA,DT-EPR,gamma,0.5,92.383,93.76,94.026,94.311,89.715,93.185
MHSA,DT-EPR,gamma,0.75,94.117,95.4,95.691,96.196,92.422,94.921
MHSA,DT-EPR,gamma,0.9,92.758,94.469,94.886,95.664,91.061,93.864
MHSA,DT-EPR,p,0.1,86.937,89.188,89.499,89.852,82.224,88.202
MHSA,DT-EPR,p,0.25,70.519,73.811,74.356,75.016,60.246,72.463
MHSA,DT-EPR,p,0.5,45.127,48.415,49.176,50.241,29.892,47.306
MHSA,DT-EPR,p,0.75,19.874,22.919,23.769,25.147,8.005,22.13
MHSA,DT-EPR,p,0.9,5.779,7.527,8.334,9.795,1.209,7.484
MHSA,DT-EPR,rho,0.1,72.238,77.129,78.514,80.92,65.555,75.431
MHSA,DT-EPR,rho,0.25,83.24,85.433,85.891,86.75,77.717,84.643
MHSA,DT-EPR,rho,0.5,77.065,79.541,79.939,80.41,68.736,78.525
MHSA,DT-EPR,rho,0.75,69.784,72.452,72.914,73.506,59.043,71.403
MHSA,DT-EPR,rho,0.9,64.698,67.334,67.838,68.495,52.617,66.339
Mamba,DT-EPR,gamma,0.1,59.05,61.932,62.406,63.056,45.737,60.835
Mamba,DT-EPR,gamma,0.25,80.927,83.085,83.396,83.775,73.775,82.188
Mamba,DT-EPR,gamma,0.5,92.351,93.716,94.0,94.272,89.652,93.144
Mamba,DT-EPR,gamma,0.75,93.902,95.155,95.429,95.957,92.042,94.686
Mamba,DT-EPR,gamma,0.9,92.335,94.14,94.596,95.331,90.026,93.484
Mamba,DT-EPR,p,0.1,86.881,89.115,89.463,89.821,82.222,88.151
Mamba,DT-EPR,p,0.25,70.564,73.746,74.253,74.858,60.243,72.445
Mamba,DT-EPR,p,0.5,45.126,48.322,48.892,49.775,29.801,47.191
Mamba,DT-EPR,p,0.75,20.232,23.016,23.584,24.53,7.967,22.213
Mamba,DT-EPR,p,0.9,6.496,8.055,8.534,9.429,1.16,7.896
Mamba,DT-EPR,rho,0.1,71.792,76.81,78.345,80.71,64.169,74.974
Mamba,DT-EPR,rho,0.25,83.072,85.317,85.768,86.393,77.283,84.451
Mamba,DT-EPR,rho,0.5,77.11,79.486,79.866,80.303,68.693,78.521
Mamba,DT-EPR,rho,0.75,69.819,72.427,72.824,73.353,58.984,71.383
Mamba,DT-EPR,rho,0.9,64.719,67.277,67.686,68.241,52.499,66.287
"""

    epr_data_string = """
model,dataset,Intervention,value,Acc@1,Acc@3,Acc@5,Acc@10,F1,MRR
LSTM,EPR,p,0.1,15.62,35.344,43.313,52.188,11.511,28.273
LSTM,EPR,p,0.25,9.363,20.68,26.834,34.963,6.48,17.873
LSTM,EPR,p,0.5,6.921,14.168,18.398,24.553,5.938,12.829
LSTM,EPR,p,0.75,9.337,17.347,21.51,27.404,8.984,15.465
LSTM,EPR,p,0.9,11.25,20.445,25.28,32.003,10.921,18.255
LSTM,EPR,gamma,0.1,6.969,14.486,18.819,25.195,5.717,13.081
LSTM,EPR,gamma,0.25,7.326,16.571,21.758,29.213,5.033,14.606
LSTM,EPR,gamma,0.5,9.476,22.524,29.509,38.504,6.497,19.064
LSTM,EPR,gamma,0.75,11.081,27.035,35.336,45.368,7.937,22.313
LSTM,EPR,gamma,0.9,12.492,30.504,39.137,49.293,9.023,24.714
LSTM,EPR,rho,0.1,28.398,48.624,55.12,62.34,23.346,40.697
LSTM,EPR,rho,0.25,14.207,29.514,36.4,44.923,10.498,24.688
LSTM,EPR,rho,0.5,8.143,18.236,23.788,31.55,5.681,15.915
LSTM,EPR,rho,0.75,6.24,13.638,17.99,24.404,4.756,12.301
LSTM,EPR,rho,0.9,5.929,12.414,16.15,21.924,4.846,11.337
MHSA,EPR,p,0.1,12.366,34.721,44.485,55.291,10.31,26.795
MHSA,EPR,p,0.25,10.215,22.638,29.313,38.253,9.198,19.437
MHSA,EPR,p,0.5,16.323,25.246,29.164,34.449,16.18,22.695
MHSA,EPR,p,0.75,23.829,34.624,38.559,43.205,23.641,30.821
MHSA,EPR,p,0.9,26.158,38.878,43.61,49.376,25.875,34.395
MHSA,EPR,gamma,0.1,13.502,22.429,26.818,32.89,13.311,20.15
MHSA,EPR,gamma,0.25,8.247,18.263,23.995,32.099,7.53,16.126
MHSA,EPR,gamma,0.5,7.33,21.182,29.272,40.131,6.043,17.851
MHSA,EPR,gamma,0.75,8.44,25.886,35.69,47.547,6.765,20.992
MHSA,EPR,gamma,0.9,9.478,29.162,39.721,52.186,7.66,23.247
MHSA,EPR,rho,0.1,26.087,49.595,57.699,65.897,23.527,40.379
MHSA,EPR,rho,0.25,13.119,30.416,38.382,48.027,11.337,24.898
MHSA,EPR,rho,0.5,9.19,20.451,26.607,34.952,8.428,17.7
MHSA,EPR,rho,0.75,10.617,19,23.409,29.747,10.347,17.115
MHSA,EPR,rho,0.9,12.03,19.71,23.438,28.824,11.855,17.875
Mamba,EPR,p,0.1,13.804,34.764,43.634,53.761,11.136,27.39
Mamba,EPR,p,0.25,9.815,21.915,28.453,37.22,8.244,18.875
Mamba,EPR,p,0.5,14.674,23.593,27.729,33.364,14.392,21.171
Mamba,EPR,p,0.75,23.533,34.273,38.254,43.205,23.089,30.575
Mamba,EPR,p,0.9,27.589,40.102,44.698,50.403,26.988,35.73
Mamba,EPR,gamma,0.1,12.402,21.25,25.708,31.938,12.034,19.062
Mamba,EPR,gamma,0.25,7.812,17.542,23.064,31.075,6.677,15.509
Mamba,EPR,gamma,0.5,8.086,21.379,29.089,39.383,6.235,18.208
Mamba,EPR,gamma,0.75,9.37,25.797,35.042,46.312,7.428,21.314
Mamba,EPR,gamma,0.9,10.9,29.313,38.986,50.7,8.45,23.839
Mamba,EPR,rho,0.1,22.927,44.93,52.342,60.907,20.51,36.487
Mamba,EPR,rho,0.25,13.534,29.485,37.262,46.675,11.091,24.584
Mamba,EPR,rho,0.5,8.777,19.727,25.757,34.025,7.519,17.124
Mamba,EPR,rho,0.75,9.448,17.721,22.297,28.811,8.915,15.97
Mamba,EPR,rho,0.9,10.725,18.413,22.283,27.854,10.406,16.633
"""

    # 2. 将字符串数据读入Pandas DataFrame
    dtepr_df = pd.read_csv(io.StringIO(dtepr_data_string))
    epr_df = pd.read_csv(io.StringIO(epr_data_string))

    # 3. 对每个DataFrame应用高亮逻辑
    dtepr_df_processed = apply_highlighting(dtepr_df)
    epr_df_processed = apply_highlighting(epr_df)

    # 4. 为每个处理后的DataFrame生成LaTeX表格代码
    latex_table_dtepr = generate_latex_table(dtepr_df_processed)
    latex_table_epr = generate_latex_table(epr_df_processed)
    
    # 5. 打印最终结果
    print("--- TABLE 1: DT-EPR (Corrected Script) ---")
    print(latex_table_dtepr)
    print("\n\n--- TABLE 2: EPR (Corrected Script) ---")
    print(latex_table_epr)

# Intervention result

In [ ]:

# Read the CSV files
df_epr = pd.read_csv('epr_data.csv')
df_dtepr = pd.read_csv('dtepr_data.csv')

# Defensive cleaning of model names (removes potential leading/trailing spaces)
df_epr['model'] = df_epr['model'].str.strip()
df_dtepr['model'] = df_dtepr['model'].str.strip()

# Generate plots for each dataset with the corrected function
generate_comparison_plots(df_epr, 'EPR')
generate_comparison_plots(df_dtepr, 'DT-EPR')

# Computational performance tset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. Load and Process Data ---

df = pd.read_csv("speed_test.csv")
# Define the batch size
batch_size = 16

# Calculate throughput metrics
df['Inference Throughput (samples/sec)'] = batch_size / (df['Inference (ms/batch)'] / 1000)
df['Training Throughput (samples/sec)'] = batch_size / (df['Train (ms/batch)'] / 1000)

# Prepare data for plotting
models = df['Model'].unique()
data_by_model = {model: df[df['Model'] == model].sort_values('SeqLen') for model in models}
colors = {'MHSA': 'tab:blue', 'Mamba': 'tab:green', 'LSTM': 'tab:orange'}
markers = {'LSTM': 's', 'MHSA': 'o', 'Mamba': 'o'}

# --- 2. Create the Plots ---
# Create a 2x2 subplot grid
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Computational Performance Comparison on Different Sequence Lengths', fontsize=20, fontweight='bold')

# --- Plot 1 (Top-Left): FLOPs vs. Sequence Length ---
ax1 = axes[0, 0]
for model in models:
    model_data = data_by_model[model]
    ax1.plot(model_data['SeqLen'], model_data['FLOPs (M)'],
             marker=markers[model], color=colors[model], label=model)
ax1.set_title('FLOPs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sequence Length (SeqLen)')
ax1.set_ylabel('FLOPs (M)')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.5)

# --- Plot 2 (Top-Right): Training Throughput vs. Sequence Length ---
ax2 = axes[0, 1]
for model in models:
    model_data = data_by_model[model]
    ax2.plot(model_data['SeqLen'], model_data['Training Throughput (samples/sec)'],
             marker=markers[model], color=colors[model], label=model)
ax2.set_title('Training Throughput', fontsize=14, fontweight='bold')
ax2.set_xlabel('Sequence Length (SeqLen)')
ax2.set_ylabel('Throughput (samples/sec)')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

# --- Plot 3 (Bottom-Left): Inference Throughput vs. Sequence Length ---
ax3 = axes[1, 0]
for model in models:
    model_data = data_by_model[model]
    ax3.plot(model_data['SeqLen'], model_data['Inference Throughput (samples/sec)'],
             marker=markers[model], color=colors[model], label=model)
ax3.set_title('Inference Throughput', fontsize=14, fontweight='bold')
ax3.set_xlabel('Sequence Length (SeqLen)')
ax3.set_ylabel('Throughput (samples/sec)')
ax3.legend()
ax3.grid(True, linestyle='--', alpha=0.5)

# --- Plot 4 (Bottom-Right): Inference Latency vs. Sequence Length ---
ax4 = axes[1, 1]
for model in models:
    model_data = data_by_model[model]
    ax4.plot(model_data['SeqLen'], model_data['Inference (ms/batch)']/16,
             marker=markers[model], color=colors[model], label=model)
ax4.set_title('Inference Latency', fontsize=14, fontweight='bold')
ax4.set_xlabel('Sequence Length (SeqLen)')
ax4.set_ylabel('Latency (ms/sample)')
ax4.legend()
ax4.grid(True, linestyle='--', alpha=0.5)

# --- 3. Display the Plots ---
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to make room for the suptitle
plt.show()